# Skin Mamba Lab · Kaggle T4

选择 GPU T4。先按真实路径修改下一格，复制代码到可写目录。PH² 全200张仅测试。ISIC2018→ISIC2017测试有532/600重叠，报告同时输出68张排重结果。完整训练可能跨Kaggle会话；每epoch保存last.pt。

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
CODE_INPUT = Path('/kaggle/input/your-code/skin_mamba_lab')
DATA_ROOT = Path('/kaggle/input/your-data/data')
WORK = Path('/kaggle/working/skin_mamba_lab')
assert (CODE_INPUT / 'skinmamba').is_dir(), '请设置真实CODE_INPUT'
assert (DATA_ROOT / 'isic2018/train/images').is_dir(), '请设置真实DATA_ROOT'
if not WORK.exists():
    shutil.copytree(CODE_INPUT, WORK)
os.chdir(WORK)
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], check=True)
print('code:',WORK,'data:',DATA_ROOT)

## 安装官方依赖并验证GPU
安装脚本保留Kaggle自带torch，可能需要编译CUDA扩展。失败时查看reports/kaggle_install.json，不能在缺少kernel时开始正式实验。默认FP32；若使用AMP，先另外执行verify_cuda.py --precision all。

In [ ]:
run('scripts/install_kaggle.py')
run('scripts/verify_cuda.py','--precision','fp32')

In [ ]:
run('-m','skinmamba','audit','--data-root',DATA_ROOT)

## 源域训练
默认ISIC2018 baseline，250 epoch，仅源域val选模。先探索时可设置所有比较组统一50 epoch，但之后正式实验需新目录重训。将CONFIG改为configs/ablations/geometry.yaml可跑候选机制。

In [ ]:
CONFIG = 'configs/baseline_isic2018.yaml'
RUN_DIR = Path('/kaggle/working/runs/isic2018_baseline_s42')
RESUME = None  # 新会话可填 /kaggle/input/previous-output/.../last.pt
cmd = ['-m','skinmamba','train','--config',CONFIG,'--data-root',DATA_ROOT,'--run-dir',RUN_DIR]
if RESUME:
    cmd += ['--resume',RESUME]
run(*cmd)

## 冻结方案后做最终测试
下面对best.pt自动评估ISIC2017、ISIC2018、PH²，生成CSV/JSON和可选预测掩码。不要用test评分选择配置。

In [ ]:
run('-m','skinmamba','evaluate','--checkpoint',RUN_DIR/'best.pt','--data-root',DATA_ROOT,'--save-predictions')
import pandas as pd
pd.read_csv(RUN_DIR/'evaluation/summary.csv')

In [ ]:
run('-m','skinmamba','aggregate','--root','/kaggle/working/runs','--output','/kaggle/working/runs/aggregate.csv')

结束前 Save Version 保存输出，保留last.pt、best.pt、配置、数据审计和逐图结果。当前代码的CPU测试不能代表T4测试通过；实际cuda_verification.json及profile.json是GPU核验依据。